In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma, FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=200,
#     chunk_overlap=50,
# )

loader = UnstructuredFileLoader("./files/chapter_one.txt")

# docs = loader.load()
# splitter.split_documents(docs)
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


AIMessage(content='Victory Mansions is the dilapidated apartment block where Winston lives. The entrance hall smells of boiled cabbage and old rag mats, and there is a large poster of Big Brother with eyes that seem to follow you. The lift seldom works, and the electricity is cut off during daylight hours as part of an economy drive for Hate Week. Winston’s flat is seven flights up, reached by stairs, contributing to the overall bleak, oppressive atmosphere.')

## Stuff LCEL

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma, FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=200,
#     chunk_overlap=50,
# )

loader = UnstructuredFileLoader("./files/chapter_one.txt")

# docs = loader.load()
# splitter.split_documents(docs)
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    ("human", "{question}")
])

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

chain.invoke("Descirbe Victory Mansions")

## Map Reduce LCEL

In [12]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=200,
#     chunk_overlap=50,
# )

loader = UnstructuredFileLoader("./files/chapter_one.txt")

# docs = loader.load()
# splitter.split_documents(docs)
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

map_doc_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Use the following portiono of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim.
        ------
        {context}
        """,
    ),
    ("human", "{question}")    
])
map_doc_chain = map_doc_prompt | llm

def map_docs(inputs):
    documents = inputs['documents']
    question = inputs['question']
    # results = []

    # for document in documents:
    #     result = map_doc_chain.invoke({
    #         "context": document.page_content,
    #         "question": question
    #     }).content
    #     results.append(result)

    # results = "\n\n".join(results)

    # return results

    return "\n\n".join(
        map_doc_chain.invoke(
            {"context": document.page_content, "question": question}
        ).content
        for document in documents
    )
    
map_chain = {"documents": retriever, "question": RunnablePassthrough()} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Given the following extracted parts of a long document and a question, create a final answer. 
        If you don't know the answer, just say that you don't know. Don't try to make up an answer.
        ------
        {context}
        """,
    ),
    ("human", "{question}")
])

chain = { "context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

chain.invoke("Descirbe Victory Mansions")

AIMessage(content='Victory Mansions is the grim, government-provided apartment block where Winston Smith lives in 1984. It embodies the bleak, cramped, surveilled housing that characterizes Life under the Party.\n\nKey details:\n- Exterior and location: A tall, worn block in a dreary urban setting. The area around it is rundown, with hallways smelling of boiled cabbage and old rag mats. It’s part of the economy drive and the regime’s austerity.\n- Access and condition: The block is seven flights up; the elevator (lift) rarely works, and the stairs are the norm. The surrounding architecture is imposing enough that, from the roof, you can see all four Victory Mansions blocks.\n- Interior feel: The living space is small, plain, and functional, filled with omnipresent Party paraphernalia (Victory branding, such as Victory Gin and Victory Cigarettes). The atmosphere inside is modest and oppressive rather than comfortable.\n- The telescreen: In Winston’s flat, the telescreen is mounted unusu